# Predict withheld Mucin 2 and Glucagon from shared markers

Train independent binary classifiers for **measured marker positivity**, not exclusive lineage labels. Both hidden markers are removed from the inputs everywhere; the seven shared, non-exclusive binary channels are preserved. Compare center-only predictors with exact ring fractions through radii 1–4, each with and without the center cell's measured Gaussian curvature. Curvature-assisted models are diagnostic upper-reference models: using their imputed markers for curvature-interaction analysis would introduce circularity.

Only organoids in which both stains were measured contribute supervised labels. Missing stains are recorded as unmeasured, never negative. Five outer organoid folds hold out test organoids; each development fold reserves separate calibration organoids. All feature configurations share these memberships. Gradient boosting uses a fixed tree budget without cell-wise early stopping, followed by sigmoid calibration at natural prevalence. High-precision decision thresholds are chosen on calibration organoids and evaluated only on outer test organoids.

This notebook saves fitted models, exact inputs and memberships, out-of-fold predictions and per-organoid calls. The companion evaluation notebook compares discrimination, achieved precision/recall, false positives and count errors. No imputation is applied to the unstained dataset.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Configuration

All timepoints and sphericities are included after blacklisting: predicting marker identity should not be restricted to developed crypts. Curvature inputs are unaltered exported measurements; no cohort-wide curvature outlier cleanup is fitted. Change `CURVATURE_INDICES` to [1] or [0, 1] for future mean-curvature experiments. Measured labels are the processed binary stains from the original organograph graphs, before exclusivity.

In [2]:
# Data and input definitions
DATASET = 'after_cleanup'  # Shared, non-exclusive GNN-compatible data; no restored/exclusive input dataset.
TARGET_MARKERS = ['Mucin 2', 'Glucagon']  # Independently predicted measured binary stains.
TIMEPOINTS = None  # Shared data selection; None includes every available timepoint.
BLACKLIST = True  # Exclude organoids flagged in the source dataset.
RADII = [0, 1, 2, 3, 4]  # Center only, then shared-marker fractions in every exact ring through this radius.
CURVATURE_VARIANTS = [False, True]  # Without or with measured center curvature; no neighboring curvature inputs.
CURVATURE_INDICES = [0]  # 0: Gaussian; 1: mean; these are input channels, not classification targets.

# Organoid splits
N_FOLDS = 5  # Outer organoid folds; every measured organoid is tested exactly once.
CALIBRATION_FRACTION = .25  # Fraction of each outer development set reserved for calibration and threshold selection.
SEED = 42  # Fixed initialization and organoid splitting seed shared by all configurations.

# Gradient boosting
MODEL_SETTINGS = dict(
    max_iter=120,  # Fixed boosting iterations; no cell-level validation split or early stopping.
    learning_rate=.08,  # Contribution of each boosting iteration.
    max_leaf_nodes=15,  # Maximum leaves per tree.
    min_samples_leaf=40,  # Minimum training cells in each leaf.
    l2_regularization=10.,  # Penalize large leaf values.
    balanced=True,  # Balance the training loss; calibrate afterward using natural marker prevalence.
)

# Conservative calling thresholds
PRECISION_TARGETS = [.90, .95]  # Desired calibration precision; held-out precision is measured, not guaranteed.
MIN_CALIBRATION_CALLS = 20  # Minimum predicted-positive calibration cells supporting a threshold.
MIN_CALIBRATION_ORGANOIDS = 5  # Minimum distinct calibration organoids supporting those calls.

# Execution and outputs
RUN_TRAINING = True  # Fit missing classifiers; use the companion notebook to analyze completed results.
TAG = 'shared_panel_r0to4_gaussian'  # Purpose label for this marker-prediction run.
RESUME_RUN = None  # Existing folder name under training_results/marker_prediction_training; None creates a new run.


## Load and validate node-aligned features and stains

Read the original organograph files referenced by each GNN graph. Verify original cell IDs, mesh projection IDs, shared marker values and graph edges. Construct marker fractions separately for each exact graph-distance ring. No curvature, target-marker values, ring counts, crypt labels, organoid identifiers or global shape summaries enter the fate-only design matrix. Empty rings are represented by zero fractions.

Save both measured and unstained organoids for support diagnostics; only the measured cohort is eligible for supervised fitting. Source paths and input-file hashes are recorded to make this export reproducible.

In [3]:
import json
import hashlib
import time
import gc
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from src.data.io import load_graph_dataset_from_dir
from src.data.metadata import attach_metadata_to_graphs, load_aux_metadata_for_dir, load_marker_names_from_dir
from src.data.filters import filter_graphs_by_timepoints, filter_graphs_by_blacklist, load_graph_blacklist_from_dir
from src.artifacts.paths import training_run_path
from src.marker_prediction.features import measured_labels, ring_features, select_features
from src.marker_prediction.classifiers import organoid_splits, fit_classifier, probabilities, precision_threshold
from src.marker_prediction.evaluation import classification_metrics, organoid_calls

RUN_DIR = PROJECT_ROOT / 'training_results/marker_prediction_training' / RESUME_RUN if RESUME_RUN else training_run_path(PROJECT_ROOT, 'marker_prediction_training', TAG)
settings = dict(dataset=DATASET, target_markers=TARGET_MARKERS, timepoints=TIMEPOINTS, blacklist=BLACKLIST,
    radii=RADII, curvature_variants=CURVATURE_VARIANTS, curvature_indices=CURVATURE_INDICES,
    n_folds=N_FOLDS, calibration_fraction=CALIBRATION_FRACTION, seed=SEED,
    model=MODEL_SETTINGS, precision_targets=PRECISION_TARGETS,
    min_calibration_calls=MIN_CALIBRATION_CALLS, min_calibration_organoids=MIN_CALIBRATION_ORGANOIDS)
if RESUME_RUN:
    assert json.loads((RUN_DIR / 'settings.json').read_text()) == settings, 'Changed settings require a new run.'
else:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR / 'settings.json').write_text(json.dumps(settings, indent=2))
for folder in ('models', 'predictions', 'tables'):
    (RUN_DIR / folder).mkdir(exist_ok=True)
print('Run directory:', RUN_DIR)
if (RUN_DIR / 'features.npz').exists():
    packed = np.load(RUN_DIR / 'features.npz')
    fates, curvature, labels, graph_ids = [packed[k] for k in ('fates', 'curvature', 'labels', 'graph_ids')]
    cohort = pd.read_csv(RUN_DIR / 'cohort.csv')
    marker_names = json.loads((RUN_DIR / 'shared_markers.json').read_text())
else:
    data_dir = PROJECT_ROOT / 'training_data' / DATASET
    graphs = load_graph_dataset_from_dir(str(data_dir), strict=True, target_indices=CURVATURE_INDICES)
    attach_metadata_to_graphs(graphs, load_aux_metadata_for_dir(str(data_dir)), exclude_keys=None)
    graphs = filter_graphs_by_timepoints(graphs, TIMEPOINTS, print_summary=True)
    if BLACKLIST:
        graphs = filter_graphs_by_blacklist(graphs, load_graph_blacklist_from_dir(data_dir), print_summary=True)
    marker_names = load_marker_names_from_dir(str(data_dir))
    assert not set(marker_names) & set(TARGET_MARKERS)
    feature_parts, curvature_parts, label_parts, id_parts, cohort_rows, sources = [], [], [], [], [], []
    offset = 0
    for gid, graph in enumerate(graphs):
        x = graph.x.numpy()
        y = measured_labels(graph, marker_names, TARGET_MARKERS)
        feature_parts.append(ring_features(x, graph.edge_index.numpy(), max(RADII)))
        curvature_parts.append(graph.y.numpy().reshape(len(x), -1))
        label_parts.append(y)
        id_parts.append(np.full(len(x), gid, dtype=np.int32))
        row = dict(graph_id=gid, organoid_str=graph.organoid_str, dataset=str(graph.meta['dataset']),
            timepoint=str(graph.meta.get('timepoint', 'unknown')), n_cells=len(x), start=offset, stop=offset+len(x),
            graph_path=str(graph.meta['graph_path']))
        for j, marker in enumerate(TARGET_MARKERS):
            assert np.all(y[:, j] >= 0) or np.all(y[:, j] == -1)
            row[f'measured_{marker}'] = bool(np.all(y[:, j] >= 0))
            row[f'positive_{marker}'] = int((y[:, j] == 1).sum()) if row[f'measured_{marker}'] else -1
        cohort_rows.append(row)
        for path in [data_dir / f'{graph.organoid_str}.npz', data_dir / f'{graph.organoid_str}_aux.json', Path(graph.meta['graph_path'])]:
            sources.append(dict(path=str(path), sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
        offset += len(x)
        if (gid+1) % 100 == 0:print(f'Prepared {gid+1}/{len(graphs)} organoids', flush=True)
    fates = np.concatenate(feature_parts)
    curvature = np.concatenate(curvature_parts)
    labels = np.concatenate(label_parts)
    graph_ids = np.concatenate(id_parts)
    cohort = pd.DataFrame(cohort_rows)
    np.savez_compressed(RUN_DIR / 'features.npz', fates=fates, curvature=curvature, labels=labels, graph_ids=graph_ids)
    cohort.to_csv(RUN_DIR / 'cohort.csv', index=False)
    (RUN_DIR / 'shared_markers.json').write_text(json.dumps(marker_names, indent=2))
    (RUN_DIR / 'source_files.json').write_text(json.dumps(sources, indent=2))
    del graphs, feature_parts, curvature_parts, label_parts, id_parts
    gc.collect()
measured = cohort[cohort[[f'measured_{m}' for m in TARGET_MARKERS]].all(axis=1)].copy()
assert len(measured) >= N_FOLDS
support = cohort.groupby(['dataset', 'timepoint']).agg(organoids=('graph_id', 'size'), cells=('n_cells', 'sum'))
display(support)
display(measured.groupby('timepoint')[[f'positive_{m}' for m in TARGET_MARKERS]].sum())
print(f'{len(measured)} measured organoids; {len(cohort)-len(measured)} unmeasured organoids excluded from supervised fitting.')


Run directory: /home/fmoller/Projects/LearningOrganoids/GraphNN/training_results/marker_prediction_training/shared_panel_r0to4_gaussian_20261006_114829


Loaded 1500 graphs; skipped 0.


Timepoint selection (all timepoints): kept 1500 / 1500 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day3p5              290    290    1.000
20250929         day4                342    342    1.000
20250929         day4p5              113    113    1.000
20250929         day4p5-more         477    477    1.000
20251201         day4p5              278    278    1.000
filter_graphs_by_blacklist(n_keys=17): kept 1483 / 1500 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day3p5              290    290    1.000
20250929         day4                330    342    0.965
20250929         day4p5              112    113    0.991
20250929         day4p5-more         475    477    0.996
20251201         day4p5              276    278    0.993


Prepared 100/1483 organoids


Prepared 200/1483 organoids


...

Prepared 1300/1483 organoids


Prepared 1400/1483 organoids


organoids   cells
dataset  timepoint                     
20250929 day3p5             290   58793
         day4               330   99094
         day4p5             112   47474
         day4p5-more        475  226711
20251201 day4p5             276  117234

,positive_Mucin 2,positive_Glucagon
timepoint,,
day3p5,1093,20
day4,2399,45
day4p5,1336,46
day4p5-more,3809,74


1207 measured organoids; 276 unmeasured organoids excluded from supervised fitting.


## Save shared train/calibration/test memberships

Stratify organoids by whether they contain either hidden marker, when support permits. Calibration organoids are distinct from training and test organoids. The same split is used for every marker, radius and curvature setting. Each test organoid appears in exactly one outer fold; no individual cell is randomly split away from its organoid.

In [4]:
splits = list(organoid_splits(measured, TARGET_MARKERS, N_FOLDS, CALIBRATION_FRACTION, SEED))
split_path = RUN_DIR / 'splits.json'
if split_path.exists():
    assert json.loads(split_path.read_text()) == splits
else:
    split_path.write_text(json.dumps(splits, indent=2))
rows = []
for split in splits:
    for role in ('train', 'calibration', 'test'):
        part = measured[measured.graph_id.isin(split[role])]
        rows.append(dict(fold=split['fold'], role=role, organoids=len(part), cells=int(part.n_cells.sum()),
            **{m:int(part[f'positive_{m}'].sum()) for m in TARGET_MARKERS}))
split_support = pd.DataFrame(rows)
split_support.to_csv(RUN_DIR / 'tables/split_support.csv', index=False)
display(split_support)

,fold,role,organoids,cells,Mucin 2,Glucagon
0,0,train,723,261095,5087,107
1,0,calibration,242,86162,1915,41
2,0,test,242,84815,1635,37
3,1,train,723,259499,5075,105
4,1,calibration,242,83791,1726,42
5,1,test,242,88782,1836,38
6,2,train,724,258438,4941,118
7,2,calibration,242,85859,1789,33
8,2,test,241,87775,1907,34
9,3,train,724,257634,5266,115


## Fit and score the complete classifier grid

For each fold and marker, fit radii 0–4 with and without curvature. Use class-balanced training to give sparse labels influence, then calibrate at the actual prevalence using sigmoid calibration on the reserved organoids. The fixed hyperparameters are not tuned on the outer test set.

Choose the highest-recall calibration threshold satisfying each requested empirical precision and minimum support. If none qualifies, the classifier makes no positive calls at that operating point. Save calibration support, achieved test metrics, counts by organoid, and all test probabilities. These thresholds do not guarantee the same precision in a different experiment.

In [5]:
if not RUN_TRAINING:
    raise RuntimeError('Set RUN_TRAINING=True to fit, or open the evaluation notebook for saved results.')
total = len(splits)*len(TARGET_MARKERS)*len(RADII)*len(CURVATURE_VARIANTS)
completed = 0
started = time.monotonic()
for split in splits:
    indices = {role: np.flatnonzero(np.isin(graph_ids, split[role])) for role in ('train', 'calibration', 'test')}
    for target_index, marker in enumerate(TARGET_MARKERS):
        y = labels[:, target_index]
        assert np.all(y[np.concatenate(list(indices.values()))] >= 0)
        for radius in RADII:
            for use_curvature in CURVATURE_VARIANTS:
                key = f"target{target_index}_r{radius}_curvature{int(use_curvature)}_f{split['fold']}"
                model_path = RUN_DIR / 'models' / f'{key}.joblib'
                result_path = RUN_DIR / 'tables' / f'{key}.json'
                if result_path.exists():
                    completed += 1
                    print(f'Reused {completed}/{total}: {marker}, r={radius}, curvature={use_curvature}, fold={split["fold"]}', flush=True)
                    continue
                began = time.monotonic()
                features = select_features(fates, curvature, len(marker_names), radius, use_curvature)
                train_ids, cal_ids, test_ids = [indices[role] for role in ('train', 'calibration', 'test')]
                model = fit_classifier(features[train_ids], y[train_ids], features[cal_ids], y[cal_ids],
                                       seed=SEED+split['fold'], **MODEL_SETTINGS)
                cal_p = probabilities(model, features[cal_ids])
                test_p = probabilities(model, features[test_ids])
                config = dict(key=key, marker=marker, radius=radius, use_curvature=use_curvature, fold=split['fold'])
                result_rows, call_rows, thresholds = [], [], {}
                for precision_target in PRECISION_TARGETS:
                    threshold = precision_threshold(y[cal_ids], cal_p, graph_ids[cal_ids], target=precision_target,
                        min_calls=MIN_CALIBRATION_CALLS, min_organoids=MIN_CALIBRATION_ORGANOIDS)
                    thresholds[str(precision_target)] = threshold
                    calibration = classification_metrics(y[cal_ids], cal_p, threshold)
                    scores = classification_metrics(y[test_ids], test_p, threshold)
                    result_rows.append(dict(**config, precision_target=precision_target,
                        threshold=float(threshold) if np.isfinite(threshold) else None,
                        calibration_precision=calibration['precision'], calibration_recall=calibration['recall'],
                        calibration_calls=calibration['calls'], **scores))
                    calls = organoid_calls(y[test_ids], test_p, graph_ids[test_ids], threshold)
                    for name, value in dict(**config, precision_target=precision_target).items():calls[name] = value
                    call_rows.append(calls)
                joblib.dump(dict(**model, config=config, thresholds=thresholds, shared_markers=marker_names,
                    curvature_indices=CURVATURE_INDICES, split=split, sklearn_version=sklearn.__version__), model_path)
                np.savez_compressed(RUN_DIR / 'predictions' / f'{key}.npz', row_indices=test_ids,
                    graph_ids=graph_ids[test_ids], labels=y[test_ids], probabilities=test_p.astype(np.float32))
                pd.concat(call_rows, ignore_index=True).to_csv(RUN_DIR / 'tables' / f'{key}_organoids.csv', index=False)
                result_path.write_text(json.dumps(result_rows, indent=2))
                completed += 1
                print(f'{completed}/{total}: {marker}, r={radius}, curvature={use_curvature}, fold={split["fold"]}; '
                      f'AP={scores["average_precision"]:.3f}; {time.monotonic()-began:.1f}s; elapsed {(time.monotonic()-started)/60:.1f} min', flush=True)
                del model, features
                gc.collect()
metrics = pd.DataFrame([row for p in sorted((RUN_DIR / 'tables').glob('target*_f*.json')) for row in json.loads(p.read_text())])
assert len(metrics) == total*len(PRECISION_TARGETS)
metrics.to_csv(RUN_DIR / 'test_metrics.csv', index=False)
organoid_scores = pd.concat([pd.read_csv(p) for p in sorted((RUN_DIR / 'tables').glob('target*_organoids.csv'))], ignore_index=True)
organoid_scores.to_csv(RUN_DIR / 'organoid_calls.csv', index=False)
(RUN_DIR / 'complete.json').write_text(json.dumps(dict(models=total, sklearn_version=sklearn.__version__), indent=2))
print('Completed marker-prediction run:', RUN_DIR)

1/100: Mucin 2, r=0, curvature=False, fold=0; AP=0.328; 0.7s; elapsed 0.0 min


2/100: Mucin 2, r=0, curvature=True, fold=0; AP=0.338; 0.7s; elapsed 0.0 min


3/100: Mucin 2, r=1, curvature=False, fold=0; AP=0.431; 0.8s; elapsed 0.0 min


4/100: Mucin 2, r=1, curvature=True, fold=0; AP=0.431; 0.9s; elapsed 0.1 min


5/100: Mucin 2, r=2, curvature=False, fold=0; AP=0.431; 1.1s; elapsed 0.1 min


...

97/100: Glucagon, r=3, curvature=False, fold=4; AP=0.072; 1.2s; elapsed 1.9 min


98/100: Glucagon, r=3, curvature=True, fold=4; AP=0.086; 1.4s; elapsed 2.0 min


99/100: Glucagon, r=4, curvature=False, fold=4; AP=0.052; 1.4s; elapsed 2.0 min


100/100: Glucagon, r=4, curvature=True, fold=4; AP=0.100; 1.5s; elapsed 2.0 min


Completed marker-prediction run: /home/fmoller/Projects/LearningOrganoids/GraphNN/training_results/marker_prediction_training/shared_panel_r0to4_gaussian_20261006_114829
